# Building a RAG System with LangChain

## Author: Jorge Soto Tripiana

# LLM with RAG on PDF Documents

**General Objective:**

Evaluate the impact of RAG in LLM responses and compare the effect different hyperparameters configurations have on the accuracy of responses generated, using a set of documents as knowledge base.


**Specifically:**

1. **RAG vs. No RAG:**
  Demonstrate the improvement in accuracy and coherence while reducing hallucinations when using RAG, compared to a traditional Zero-Shot approach.


2. **Chunking Analysis:**
  Observe how the sizes of chunks extracted from the documents affects the quality of the output.


3. **Parameter *k* Study (Retrieval):**
   Determine the optimal number of retrieved chunks (*k* = 1, 3, 5, 9) to maximize response quality, balancing the amount of context and avoiding noise.


4. **Model Comparison:**
   Compare the performance of different embedding models towards retrieval.


5. **Evaluation:**
   Implement automatic NLP evaluation metrics such as Hit Rate and Precision for the retriever to determine the performance of each configuration.

## Setting up dependencies


In [1]:
!pip install -q --upgrade huggingface_hub
!pip install -qU langchain-docling docling langchain-text-splitters
!pip install -q langchain-huggingface
!pip install -q bitsandbytes # 4bit quant
!pip install -q flash-linear-attention # attention optimizations

## Libraries


In [2]:
import torch
import os
import docling_parse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from google.colab import drive
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer, ChunkingSerializerProvider)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer
from langchain_docling.loader import DoclingLoader, ExportType
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer

In [ ]:
# gemma requires login
from huggingface_hub import login
login()

# Storing the weights in drive, avoiding redundant downloads (if ran on cloud)

In [ ]:
drive.mount("/content/drive")

MODEL_ID = "Qwen/Qwen3.5-9B"
SAVE_DIR = "/content/drive/MyDrive/Models/Qwen3.5-9B-4bit"
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

if os.path.exists(f"{SAVE_DIR}/config.json"):
    # load model
    pipe = pipeline(
        task="text-generation",
        model=SAVE_DIR,
        dtype=torch.bfloat16,
        device_map="auto",
    )
else:
    # download & store in drive
    pipe = pipeline(
        task="text-generation",
        model=MODEL_ID,
        dtype=torch.bfloat16,
        device_map="auto",
        model_kwargs={"quantization_config": BitsAndBytesConfig(
            # less vram usage & faster compute (bfloat16)
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)},
    )

    pipe.model.save_pretrained(SAVE_DIR)
    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(SAVE_DIR)

# get tokenizer
tokenizer = pipe.tokenizer

In [5]:
DOCUMENTS_PATH = [
    "f1_2025_driver_stats.pdf",
    "nurburgring_data.pdf",
    "porsche_gt3rs_992.pdf"
]


## Retrieval Evaluation

 We compare different chunking sizes, k retrieved chunks, and embedding models.

In [6]:
# ground truth set
gold = [
    # doc 1
    {
        "q": "How many points did Charles Leclerc get in the 2025 Driver Championship?",
        "doc": "f1_2025_driver_stats.pdf",
        "evidence": ["Leclerc", "242"]
    },
    {
        "q": "Who won the 2025 F1 Driver Championship and how many points did they get?",
        "doc": "f1_2025_driver_stats.pdf",
        "evidence": ["L. Norris", "423"]
    },
    {
        "q": "In which round did Max Verstappen score his highest points total, and how many points did he get?",
        "doc": "f1_2025_driver_stats.pdf",
        "evidence": ["Verstappen", "United States", "33"]
    },
    {
        "q": "How many points did Carlos Sainz score in the China round of the 2025 season?",
        "doc": "f1_2025_driver_stats.pdf",
        "evidence": ["Sainz", "China", "1"]
    },
    {
        "q": "How many points did Fernando Alonso get in the last round (Abu Dhabi) of the 2025 season?",
        "doc": "f1_2025_driver_stats.pdf",
        "evidence": ["Alonso", "Abu Dhabi", "8"]
    },

    # doc 2

    {
        "q": "When did the Nürburgring officially open?",
        "doc": "nurburgring_data.pdf",
        "evidence": ["officially opened", "18 June 1927"]
    },
    {
        "q": "Which car holds the outright lap record on the Nordschleife and what is the time?",
        "doc": "nurburgring_data.pdf",
        "evidence": ["Porsche 919 Hybrid Evo", "5:19.546"]
    },
    {
        "q": "Who holds the F1 lap record on the GP-Strecke and what is the time?",
        "doc": "nurburgring_data.pdf",
        "evidence": ["Max Verstappen", "1:28.139"]
    },
    {
        "q": "What should a driver do in case of an accident or breakdown during Tourist Drives?",
        "doc": "nurburgring_data.pdf",
        "evidence": ["accident or breakdown", "do not drive on or reverse"]
    },
    {
        "q": "How much does a single lap of the Nordschleife cost during Tourist Drives on a weekend in 2026?",
        "doc": "nurburgring_data.pdf",
        "evidence": ["Nordschleife lap", "EUR 35"]
    },

    # doc 3

    {
        "q": "What type of engine does the Porsche 911 GT3 RS (992) have?",
        "doc": "porsche_gt3rs_992.pdf",
        "evidence": ["Engine", "4.0-litre naturally aspirated flat-six"]
    },
    {
        "q": "What is the 0-100 km/h acceleration time for the Porsche 911 GT3 RS (992)?",
        "doc": "porsche_gt3rs_992.pdf",
        "evidence": ["0-100 km/h", "3.2 s"]
    },
    {
        "q": "How much horsepower (hp) does the Porsche 911 GT3 RS (992) produce?",
        "doc": "porsche_gt3rs_992.pdf",
        "evidence": ["Power", "518 hp"]
    },
    {
        "q": "What is the kerb weight (DIN) of the Porsche 911 GT3 RS (992)?",
        "doc": "porsche_gt3rs_992.pdf",
        "evidence": ["Kerb weight (DIN)", "1,450 kg"]
    },
    {
        "q": "How much downforce does the Porsche 911 GT3 RS (992) generate at 285 km/h?",
        "doc": "porsche_gt3rs_992.pdf",
        "evidence": ["Downforce at 285 km/h", "860 kg"]
    },

    # unanswerable questions (model knowledge cutoff is previous)
    {
        "q" : "Why did Jensen Huang, the CEO of NVIDIA, resign?",
        "doc" : "",
        "evidence" : ["", ""]
    },
    {
        "q" : "What's the API pricing for Qwen4 Max?",
        "doc" : "",
        "evidence" : ["", ""]
    }
]

In [ ]:
def norm(s):
    return " ".join(s.lower().split())

def is_relevant(doc, item):
    return (doc.metadata["source"] == item["doc"]
            and all(norm(e) in norm(doc.page_content) for e in item["evidence"]))


def eval_retrieval(vs, documents, gold, ks=(1, 3, 5, 9), **cfg):
    ks = sorted(ks)
    max_k = max(ks)
    out = []

    for item in gold:
        res = vs.similarity_search(item["q"], k=max_k)
        rel = [is_relevant(d, item) for d in res]
        R = sum(is_relevant(d, item) for d in documents)
        first = next((i + 1 for i, r in enumerate(rel) if r), None)  # rank 1-based

        for k in ks:
            hits = sum(rel[:k])
            den = min(k, R)
            out.append({
                **cfg,
                "q": item["q"],
                "doc": item["doc"],
                "k": k,
                "n_rel": R,
                "answerable": R > 0,
                "is_negative": item["doc"] == "",
                "hits": hits,
                "precision": hits / k,
                "precision_norm": hits / den if den else np.nan,
                "recall": hits / R if R else np.nan,
                "rr": (1.0 / first) if (first is not None and first <= k) else 0.0,
                "hit": int(hits > 0),
            })
    return out

In [ ]:
CHUNK_TOKENIZER_ID = EMBED_MODEL_ID
embedders = {m: HuggingFaceEmbeddings(model_name=m)
             # to save IDs
             for m in ["Qwen/Qwen3-Embedding-0.6B", "google/embeddinggemma-300m"]}

In [ ]:
class MDTableProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer()
            )

providers = {"triplet": None, "markdown": MDTableProvider()}

converter = DocumentConverter()
# we will apply chunking in the loop to test different sizes
dl_docs = [converter.convert(p).document for p in DOCUMENTS_PATH]

In [ ]:
# test parameters
sizes = [128, 256, 512]
ks= (1, 3, 5, 9)

rows = []
for size in sizes:
  for name, provider in providers.items():
    kwargs = {"serializer_provider": provider} if provider else {}
    chunker = HybridChunker(tokenizer=CHUNK_TOKENIZER_ID, max_tokens=size,
                            repeat_table_header=True, **kwargs)
    documents = [
        Document(page_content=chunker.contextualize(c),
                 metadata={"source": d.origin.filename})
        for d in dl_docs for c in chunker.chunk(dl_doc=d)
    ]

    if 1: # debug
      print(f"Document Parsing with {name}\n")
      for d in documents:
          print(d.page_content, '\n')

    for emb_id, emb in embedders.items():
        vs = InMemoryVectorStore.from_documents(documents, emb)
        rows += eval_retrieval(vs, documents, gold, ks,
                               chunk_size=size, model=emb_id,
                               serializer=name)

In [ ]:
# save results in a dataframe
df = pd.DataFrame(rows)

## Visualizing Results

In [ ]:
# params
sizes = [128, 256, 512]
ks    = [1, 3, 5, 9]

# labels and colors
model_labels      = {"qwen": "Qwen", "gemma": "Gemma"}
serializer_labels = {"triplet": "Triplet", "markdown": "Markdown"}
doc_labels        = {"f1_2025_driver_stats.pdf": "F1 2025 (tables)",
                     "nurburgring_data.pdf":     "Nürburgring (text)",
                     "porsche_gt3rs_992.pdf":    "Porsche (spec sheet)"}

models    = ["Qwen", "Gemma"]
serials   = ["Triplet", "Markdown"]
doc_order = list(doc_labels.values())

size_colors = {128: "#4C72B0", 256: "#DD8452", 512: "#55A868"}
k_colors    = {1: "#4C72B0", 3: "#DD8452", 5: "#55A868", 9: "#C44E52"}
ser_colors  = {"Triplet": "#8172B3", "Markdown": "#64B5CD"}

METRIC_TITLES = {"precision": "Precision@k (raw)",
                 "precision_norm": "Normalized Precision@k",
                 "recall": "Recall@k", "hit": "Hit Rate@k", "rr": "MRR@k"}

def get_model_label(m):
    m = m.lower()
    for key, lab in model_labels.items():
        if key in m:
            return lab
    return m

df_ret = df.loc[~df["is_negative"]].copy()
df_ret["model_label"]      = df_ret["model"].apply(get_model_label)
df_ret["serializer_label"] = df_ret["serializer"].map(serializer_labels)
df_ret["doc_label"]        = df_ret["doc"].map(doc_labels)

df_ret = df_ret.loc[
    df_ret["model_label"].isin(models)
    & df_ret["chunk_size"].isin(sizes)
    & df_ret["k"].isin(ks)
].copy()



def plot_grid(data, metric, x_param, x_values,
              series_param, series_values, series_colors,
              row_param="serializer_label", row_values=serials,
              col_param="model_label",      col_values=models,
              suptitle=None, legend_title=None, ylim=(0, 1.05)):
    nrows, ncols = len(row_values), len(col_values)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows),
                             dpi=200, sharex=True, sharey=True, squeeze=False)
    x = np.arange(len(x_values))
    mtitle = METRIC_TITLES.get(metric, metric)

    for i, rv in enumerate(row_values):
        for j, cv in enumerate(col_values):
            ax = axes[i][j]
            sub = data.loc[(data[row_param] == rv) & (data[col_param] == cv)]
            for sv in series_values:
                # avg metric only inside this single cell of the grid
                y = (sub.loc[sub[series_param] == sv]
                       .groupby(x_param)[metric].mean()
                       .reindex(x_values).values)
                ax.plot(x, y, marker="o", lw=2, ms=5,
                        color=series_colors[sv], label=str(sv))

            # every panel is self-contained (safe to screenshot alone)
            ax.set_title(f"{rv}  |  {cv}\n{mtitle} vs {x_param}",
                         fontsize=11, fontweight="bold")
            ax.set_xticks(x); ax.set_xticklabels(x_values)
            ax.set_ylim(*ylim)
            ax.grid(alpha=.35, ls="--"); ax.set_axisbelow(True)
            ax.set_xlabel(x_param, fontsize=10)
            ax.set_ylabel(mtitle, fontsize=10)
            ax.tick_params(labelbottom=True, labelleft=True)  # shared axes hide them
            ax.legend(title=legend_title or series_param, loc="best",
                      fontsize=8, title_fontsize=8, framealpha=.9)

    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, .96))
    else:
        fig.tight_layout()
    plt.show()
    return fig, axes


# chunk size & k effect
def plot_heatmaps(data, metric, x_param="k", x_values=ks,
                  y_param="chunk_size", y_values=sizes,
                  row_param="serializer_label", row_values=serials,
                  col_param="model_label",      col_values=models):
    nrows, ncols = len(row_values), len(col_values)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows),
                             dpi=200, squeeze=False)
    mtitle = METRIC_TITLES.get(metric, metric)

    for i, rv in enumerate(row_values):
        for j, cv in enumerate(col_values):
            ax = axes[i][j]
            sub = data.loc[(data[row_param] == rv) & (data[col_param] == cv)]
            piv = (sub.pivot_table(index=y_param, columns=x_param,
                                   values=metric, aggfunc="mean")
                      .reindex(index=y_values, columns=x_values))
            im = ax.imshow(piv.values, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")

            # values on cells
            for a in range(len(y_values)):
                for b in range(len(x_values)):
                    v = piv.values[a, b]
                    if np.isfinite(v):
                        ax.text(b, a, f"{v:.2f}", ha="center", va="center",
                                fontsize=9, color="white" if v < .25 else "black")

            ax.set_xticks(range(len(x_values))); ax.set_xticklabels(x_values)
            ax.set_yticks(range(len(y_values))); ax.set_yticklabels(y_values)
            ax.set_xlabel(x_param); ax.set_ylabel(y_param)
            ax.set_title(f"{rv}  |  {cv}\n{mtitle} ({y_param} × {x_param})",
                         fontsize=11, fontweight="bold")

            cb = fig.colorbar(im, ax=ax, fraction=.046, pad=.04, label=mtitle)
            ax.cbar_ref = cb  # kept so save_panels can include it

    fig.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
# effcet of k
for m in ["precision_norm", "hit", "rr"]:
    plot_grid(df_ret, m, "k", ks, "chunk_size", sizes, size_colors,
              legend_title="chunk size",
              suptitle=f"{METRIC_TITLES[m]} vs top-k - per chunk size, embedder and serializer")

# effect of chunk size
for m in ["precision_norm", "hit"]:
    plot_grid(df_ret, m, "chunk_size", sizes, "k", ks, k_colors,
              legend_title="k",
              suptitle=f"{METRIC_TITLES[m]} vs chunk size - per k, embedder and serializer")

for m in ["precision_norm", "hit", "recall"]:
    plot_heatmaps(df_ret, m)

# serializers effect, per document (fixed chunk size)
plot_grid(df_ret.loc[df_ret["chunk_size"] == 256], "hit", "k", ks,
          "serializer_label", serials, ser_colors,
          row_param="doc_label", row_values=doc_order,
          legend_title="table serializer",
          suptitle="Serializer effect by document (chunk size = 256)")

plot_grid(df_ret, "precision", "k", ks, "chunk_size", sizes, size_colors,
          legend_title="chunk size",
          suptitle="RAW precision@k - artificial ceiling min(k,R)/k")

## Evaluating LLM Responses (RAG vs No RAG)

In [7]:
def generate(question, context=None):
    system = ("You are a helpful assistant. Answer ONLY if you are sure. "
              "Be extremely concise. If you don't know, say 'I don't know.'")
    user = question
    if context: # RAG
        system = ("Answer the user's question using ONLY the provided context. "
                  "Be extremely concise. If the answer is not in the context, say 'I don't know.'")
        user = f"Context:\n{context}\n\nQuestion: {question}"

    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    out = pipe(prompt, return_full_text=False, max_new_tokens=64,
               do_sample=False, top_p=0.8, top_k=20,
               min_p=0.0, repetition_penalty=1.0)
    return out[0]["generated_text"].strip()

In [ ]:
CHUNK_SIZE = 256
K = 3
embedder = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m')

In [ ]:
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=CHUNK_SIZE,
    repeat_table_header=True,
    serializer_provider=MDTableProvider(),
)

# need to run cell for dl_docs definitions first
documents = [
    Document(page_content=chunker.contextualize(c),
             metadata={"source": d.origin.filename})
    for d in dl_docs for c in chunker.chunk(dl_doc=d)
]

vector_store = InMemoryVectorStore.from_documents(documents, embedder)

results = []
for item in gold:
    q = item["q"]
    chunks = vector_store.similarity_search(q, k=K)
    context = "\n\n".join(d.page_content for d in chunks)
    results.append({
        "q": q,
        "no_rag": generate(q),
        "rag": generate(q, context),
        "retrieved": [d.page_content for d in chunks],
    })



In [ ]:
for i, r in enumerate(results):
  print(f"Question {i+1}: {r['q']}\n No RAG answer: {r['no_rag']}\n RAG Answer: {r['rag']}")
  print('-'*100)

Question 1: How many points did Charles Leclerc get in the 2025 Driver Championship?
 No RAG answer: I don't know.
 RAG Answer: Charles Leclerc got 242 points in the 2025 Driver Championship.
----------------------------------------------------------------------------------------------------
Question 2: Who won the 2025 F1 Driver Championship and how many points did they get?
 No RAG answer: I don't know. The 2025 F1 season has not happened yet.
 RAG Answer: L. Norris won the 2025 F1 Driver Championship with 423 points.
----------------------------------------------------------------------------------------------------
Question 3: In which round did Max Verstappen score his highest points total, and how many points did he get?
 No RAG answer: I don't know.
 RAG Answer: Max Verstappen scored his highest points total of 25 points in Rounds 03, 07, 16, 17, 21, and 24.
----------------------------------------------------------------------------------------------------
Question 4: How many 

## Evaluating Responses
The criteria is as follows:
- 0 points: Completely incorrect answer, containing no factual information relevant to the question.

- 0 points: Refrains to answer when information is available.

- 1 point: Mostly incorrect answer, containing some relevant factual information but major errors.

- 2 points: Partially correct answer. Contains relevant and accurate information but is incomplete or has noticeable factual errors.

- 3 points: Almost fully correct answer, with only minor mistakes or omissions.

- 4 points: Fully correct and accurate answer.

- 4 points: Appropriate abstention when the question cannot be answered using the available knowledge base.

In [14]:
scores = {
    "no_rag" : [0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 3, 4, 4, 0, 1, 4, 4],  # q1..q17
    "rag" : [4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4],
}

In [ ]:
# plotting results

MAX_PER_Q = 4

labels = {"no_rag": "No RAG", "rag": "RAG"}
x = [labels.get(k, k.upper()) for k in scores.keys()]
y = [sum(p) for p in scores.values()]
max_total = len(next(iter(scores.values()))) * MAX_PER_Q

colors = ["lightcoral", "lightcyan"]

fig, ax = plt.subplots(figsize=(8, 6), dpi=200)

bars = ax.bar(x, y, color=colors, lw=1, edgecolor="black", width=0.55)

# max points
ax.axhline(max_total, color="black", ls="--", lw=1.2)
ax.text(len(x) - 0.5, max_total + 1, f"Max points: {max_total}",
        ha="right", va="bottom", fontsize=11)

ax.bar_label(
    bars,
    labels=[f"{v}\n({v / max_total:.0%})" for v in y],
    fontsize=14,
    padding=10,
)

ax.set_ylim(0, max_total * 1.12)
ax.set_ylabel("Total Points", fontsize=14)
ax.tick_params(axis="x", labelsize=14)
ax.set_title("RAG vs No RAG", fontsize=18)
ax.grid(axis="y", ls="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()



In [ ]:
# if "doc" empty in gold -> not answerable
answerable = [bool(item["doc"]) for item in gold]

ABSTAIN_MARKERS = ("i don't know", "i do not know", "cannot be answered",
                   "not in the context", "no information")

def abstained(answer):
    a = answer.lower()
    return any(m in a for m in ABSTAIN_MARKERS)

flags = {name: [abstained(r[name]) for r in results] for name in scores}
# heuristic: override manually if needed, e.g. flags["no_rag"][8] = False

def categorize(s, is_answerable, did_abstain):
    if did_abstain:
        return "Wrong abstention" if is_answerable else "Correct abstention"
    if s >= 3: return "Correct"
    if s == 2: return "Partial"
    return "Incorrect"

categories = ["Correct", "Partial", "Incorrect", "Correct abstention", "Wrong abstention"]
cat_colors = {"Correct": "palegreen", "Partial": "khaki", "Incorrect": "lightcoral",
              "Correct abstention": "lightblue", "Wrong abstention": "lightgray"}
labels = {"no_rag": "No RAG", "rag": "RAG"}

counts = {
    name: [sum(categorize(s, a, f) == c
               for s, a, f in zip(vals, answerable, flags[name]))
           for c in categories]
    for name, vals in scores.items()
}
n_q = len(next(iter(scores.values())))

fig, ax = plt.subplots(figsize=(10, 6), dpi=200)

n_groups = len(counts)
n_cats = len(categories)
group_width = 0.8
bar_width = group_width / n_cats
x = np.arange(n_groups)

for i, cat in enumerate(categories):
    vals = np.array([counts[name][i] for name in counts])
    offset = (i - (n_cats - 1) / 2) * bar_width
    bars = ax.bar(x + offset, vals, width=bar_width,
                  color=cat_colors[cat], edgecolor="black", lw=1, label=cat)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.2,
                f"{v} ({v / n_q:.0%})",
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([labels[k] for k in counts], fontsize=14)
ax.set_ylabel("Number of questions", fontsize=14)
ax.set_ylim(0, 17)
ax.yaxis.set_major_locator(MultipleLocator(1))
ax.set_title("Answer breakdown: RAG vs No RAG", fontsize=18)
ax.grid(axis="y", ls="--", alpha=0.4)
ax.set_axisbelow(True)
ax.legend(title="Answer type", loc="upper right", framealpha=0.9) # inside, top right
plt.tight_layout()
plt.show()